In [ ]:
import os
import torch
from sae_all import GatedSparseAutoencoder

os.environ["HF_HOME"] = "models"

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


In [37]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)
model = model.to(device)
model.eval()
tokenizer.pad_token = tokenizer.eos_token

W_U = model.lm_head.weight.detach()
print("Model loaded successfully")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 9056.86it/s]


Model loaded successfully


In [38]:
INPUT_DIM = 768
LATENT_DIM = 2048

sae = GatedSparseAutoencoder(input_dim=INPUT_DIM, latent_dim=LATENT_DIM).to(device)
sae.load_state_dict(torch.load("models/sae_trained/gated_sae.pt", map_location=device))
sae.eval()
print("Loaded sae weights successfully")

Loaded sae weights successfully


In [39]:
class ConceptIntervention:
    def __init__(self, model, layer_idx, direction, alpha=0.0, mode="subtract"):
        self.model = model
        self.layer_idx = layer_idx
        self.direction = F.normalize(direction, dim=-1)
        self.alpha = alpha
        self.mode = mode
        self.hook_handle = None

    def __enter__(self):
        target_layer = self.model.transformer.h[self.layer_idx]
        
        def hook_fn(module, args, output):
            if isinstance(output, tuple):
                h = output[0]
                rest = output[1:]
            else:
                h = output
                rest = None

            if self.mode == "project":
                proj = torch.sum(h * self.direction, dim=-1, keepdim=True) * self.direction
                h_mod = h - proj
            else:
                h_mod = h - self.alpha * self.direction

            if rest is not None:
                return (h_mod,) + rest
            return h_mod

        self.hook_handle = target_layer.register_forward_hook(hook_fn)
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.hook_handle is not None:
            self.hook_handle.remove()

In [40]:
def generate_text(prompt, max_new_tokens=15, feature_id=None, alpha=0.0, mode="subtract"):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    if feature_id is not None:
        direction = sae.decoder.weight[:, feature_id].detach()
        with ConceptIntervention(model, layer_idx=6, direction=direction, alpha=alpha, mode=mode):
            output_ids = model.generate(
                input_ids,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False
            )
    else:
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=False
        )
        
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

def compare_next_token_probs(prompt, feature_id, alpha=5.0, top_k=6):
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    direction = sae.decoder.weight[:, feature_id].detach()

    with torch.no_grad():
        clean_logits = model(input_ids).logits[:, -1, :][0]
        clean_probs = F.softmax(clean_logits, dim=-1)

    with torch.no_grad():
        with ConceptIntervention(model, layer_idx=6, direction=direction, mode="project"):
            ablated_logits = model(input_ids).logits[:, -1, :][0]
            ablated_probs = F.softmax(ablated_logits, dim=-1)

    with torch.no_grad():
        with ConceptIntervention(model, layer_idx=6, direction=direction, alpha=alpha, mode="subtract"):
            supp_logits = model(input_ids).logits[:, -1, :][0]
            supp_probs = F.softmax(supp_logits, dim=-1)

    top_clean_probs, top_clean_ids = torch.topk(clean_probs, k=top_k)
    
    rows = []
    for p, idx in zip(top_clean_probs, top_clean_ids):
        tok_str = repr(tokenizer.decode([idx.item()]))
        rows.append({
            "token": tok_str,
            "clean_prob": round(p.item(), 4),
            "ablated_prob": round(ablated_probs[idx].item(), 4),
            "suppressed_prob": round(supp_probs[idx].item(), 4)
        })
        
    return pd.DataFrame(rows)

In [41]:
prompt_329 = "I am planning a trip to New"
df_probs_329 = compare_next_token_probs(prompt_329, feature_id=329, alpha=50.0, top_k=6)
print(f"Prompt: {repr(prompt_329)}")
print("Next-token probability shift when suppressing Feature #329:")
print(df_probs_329)

print("\nGeneration - Baseline:")
print(generate_text(prompt_329, max_new_tokens=10))

print("\nGeneration - Suppressed Feature #329")
print(generate_text(prompt_329, max_new_tokens=10, feature_id=329, alpha=50.0))

print("\nGeneration - Fully Ablated (Orthogonal projection):")
print(generate_text(prompt_329, max_new_tokens=10, feature_id=329, mode="project"))

Prompt: 'I am planning a trip to New'
Next-token probability shift when suppressing Feature #329:
          token  clean_prob  ablated_prob  suppressed_prob
0       ' York'      0.5593        0.1390           0.0302
1    ' Zealand'      0.1686        0.1100           0.0124
2    ' Orleans'      0.0807        0.0360           0.0140
3     ' Jersey'      0.0338        0.0095           0.0018
4     ' Mexico'      0.0326        0.0308           0.0090
5  ' Hampshire'      0.0297        0.0111           0.0026

Generation - Baseline:
I am planning a trip to New York City to visit my family and friends. I

Generation - Suppressed Feature #329
I am planning a trip to New Belgium, where I will be staying for the next

Generation - Fully Ablated (Orthogonal projection):
I am planning a trip to New York to visit my family and friends. I am


In [42]:
prompt_1378 = "The rally gathered thousands of passionate"
df_probs_1378 = compare_next_token_probs(prompt_1378, feature_id=1378, alpha=50.0, top_k=6)
print(f"Prompt: {repr(prompt_1378)}")
print("Next-token probability shift when suppressing Feature #1378:")
print(df_probs_1378)

print("\nGeneration - Baseline:")
print(generate_text(prompt_1378, max_new_tokens=12))

print("\nGeneration - Suppressed Feature #1378:")
print(generate_text(prompt_1378, max_new_tokens=12, feature_id=1378, alpha=50.0))

print("\nGeneration - Fully Ablated (Orthogonal projection):")
print(generate_text(prompt_1378, max_new_tokens=12, feature_id=1378, mode="project"))

Prompt: 'The rally gathered thousands of passionate'
Next-token probability shift when suppressing Feature #1378:
           token  clean_prob  ablated_prob  suppressed_prob
0  ' supporters'      0.4046        0.2695           0.0031
1         ' and'      0.0532        0.0662           0.0675
2      ' people'      0.0499        0.0401           0.0135
3            ','      0.0471        0.0625           0.1181
4        ' fans'      0.0328        0.1260           0.0525
5  ' protesters'      0.0306        0.0135           0.0001

Generation - Baseline:
The rally gathered thousands of passionate supporters, many of whom were wearing the same red and white

Generation - Suppressed Feature #1378:
The rally gathered thousands of passionate, well-dressed, and well-dressed guests

Generation - Fully Ablated (Orthogonal projection):
The rally gathered thousands of passionate supporters, many of whom were in attendance.

"


In [44]:
prompt_393 = "The attorney presented crucial evidence before the federal"
df_probs_393 = compare_next_token_probs(prompt_393, feature_id=393, alpha=50.0, top_k=6)
print(f"Prompt: {repr(prompt_393)}")
print("Next-token probability shift when suppressing Feature #393:")
print(df_probs_393)

print("\nGeneration - Baseline:")
print(generate_text(prompt_393, max_new_tokens=12))

print("\nGeneration - Suppressed Feature #393:")
print(generate_text(prompt_393, max_new_tokens=12, feature_id=393, alpha=50.0))

print("\nGeneration - Fully Ablated (Orthogonal projection):")
print(generate_text(prompt_393, max_new_tokens=12, feature_id=393, mode="project"))

Prompt: 'The attorney presented crucial evidence before the federal'
Next-token probability shift when suppressing Feature #393:
        token  clean_prob  ablated_prob  suppressed_prob
0    ' court'      0.3517        0.1889           0.0005
1    ' judge'      0.1726        0.1116           0.0003
2    ' grand'      0.0776        0.1324           0.0021
3     ' jury'      0.0709        0.0338           0.0001
4  ' appeals'      0.0402        0.0273           0.0001
5    ' panel'      0.0365        0.0466           0.0010

Generation - Baseline:
The attorney presented crucial evidence before the federal court.

"The defendant's conduct was not a

Generation - Suppressed Feature #393:
The attorney presented crucial evidence before the federal government's security forces, including the use of a "H

Generation - Fully Ablated (Orthogonal projection):
The attorney presented crucial evidence before the federal court.

"The evidence is that the defendant was
